# ABSA Pipeline: Aspect-Based Sentiment Analysis
## Dataset: IMDb Reviews in Spanish (50,000 filas)

**Arquitectura:**
- **SpaCy** (`es_core_news_md`): Extracción sintáctica de pares (Aspecto, Adjetivo)
- **BERT Fine-tuning**: Ajustar `pysentimiento/robertuito-sentiment-analysis` al dominio IMDb
- **BERT Inferencia**: Clasificación de sentimiento por fragmento con modelo fine-tuneado
- **Output**: Parquet estructurado + pesos del modelo fine-tuneado

---

### Celda 1: Instalación e Importación de Librerías

**Objetivo:** Instalar de forma silenciosa todas las dependencias necesarias y descargar el modelo de SpaCy en español.

**Buenas prácticas aplicadas:**
- Uso de `-q` para suprimir logs innecesarios de pip
- Instalación de `pyarrow` para manejo eficiente de archivos Parquet
- Instalación de `datasets`, `accelerate`, `evaluate` para fine-tuning con Hugging Face Trainer
- Descarga del modelo `es_core_news_md` (balance entre precisión sintáctica y velocidad)

In [ ]:
# Celda 1: Instalación de Dependencias
# Ejecutar UNA SOLA VEZ al inicio de la sesión de Colab

!pip install -q pandas pyarrow spacy transformers torch tqdm kaggle datasets accelerate evaluate

# Descargar modelo de SpaCy para español (medium = buen balance velocidad/precisión)
!python -m spacy download es_core_news_md -q

print("✅ Instalación completada. Reiniciar runtime si se solicita.")

In [ ]:
# Celda 1 (continuación): Importaciones

import os
import sys
import gc
import logging
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np
import torch
import spacy
from tqdm.notebook import tqdm

# Agregar src/ al path para importar módulos locales
sys.path.insert(0, str(Path.cwd() / "src"))

from text_processing import (
    cargar_modelo_spacy,
    extraer_aspectos_adjetivos,
    procesar_textos_spacy,
    extraer_fragmentos_para_bert
)
from sentiment_analysis import (
    configurar_pipeline_sentimiento,
    analizar_sentimiento_lote,
    mapear_sentimientos
)

# Verificar disponibilidad de GPU
print(f"🔥 PyTorch versión: {torch.__version__}")
print(f"🚀 GPU disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"📊 GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ Advertencia: No se detectó GPU. El procesamiento será más lento.")

### Celda 2: Configuración del Entorno

**Objetivo:** Montar Google Drive, descargar el dataset desde Kaggle y configurar rutas y logging.

**Estrategia de datos:**
- Montar Google Drive para persistir checkpoints, resultados y pesos del modelo
- Descargar dataset usando API de Kaggle (requiere `kaggle.json` en Drive)
- Cargar CSV con tipos optimizados (`category` para sentimiento)
- **Muestreo configurable**: `SAMPLE_SIZE` permite probar con subsets (ej: 1000) antes de procesar las 50k filas

**Manejo de excepciones:**
- Si Kaggle falla, intentar cargar desde Drive directamente
- Validar que las columnas requeridas existan (`review_es`, `sentiment`)

In [ ]:
# Celda 2: Montar Google Drive y Configurar Rutas

from google.colab import drive

# Montar Google Drive (requiere autorización)
drive.mount('/content/drive')

# Rutas del proyecto (ajustar según tu estructura en Drive)
BASE_PATH = Path("/content/drive/MyDrive/proyecto-absa-imdb")
DATA_RAW = BASE_PATH / "data" / "raw"
DATA_PROCESSED = BASE_PATH / "data" / "processed"
DATA_MODELS = BASE_PATH / "data" / "models"
LOGS_PATH = BASE_PATH / "logs"

# Crear directorios si no existen
for path in [DATA_RAW, DATA_PROCESSED, DATA_MODELS, LOGS_PATH]:
    path.mkdir(parents=True, exist_ok=True)
    print(f"📁 {path}")

print("\n✅ Entorno configurado correctamente.")

In [ ]:
# Celda 2: Configuración de Logging

log_file = LOGS_PATH / f"absa_pipeline_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(log_file),
        logging.StreamHandler(sys.stdout)
    ]
)

logger = logging.getLogger(__name__)
logger.info("🚀 Pipeline ABSA iniciado")

In [ ]:
# Celda 2: Descarga del Dataset desde Kaggle

import zipfile

KAGGLE_DATASET = "lucamla/imdb-reviews-in-spanish"  # Ajustar si cambia el slug
CSV_FILENAME = "IMDB Dataset Spanish.csv"

csv_path = DATA_RAW / CSV_FILENAME

if not csv_path.exists():
    logger.info("📥 Descargando dataset desde Kaggle...")
    
    # Configurar API de Kaggle (asume kaggle.json en /content/drive/MyDrive/)
    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    !cp /content/drive/MyDrive/kaggle.json ~/.kaggle/kaggle.json 2>/dev/null
    !chmod 600 ~/.kaggle/kaggle.json
    
    # Descargar dataset
    !kaggle datasets download -d {KAGGLE_DATASET} -p {DATA_RAW}
    
    # Descomprimir si viene en zip
    zip_files = list(DATA_RAW.glob("*.zip"))
    if zip_files:
        with zipfile.ZipFile(zip_files[0], 'r') as zip_ref:
            zip_ref.extractall(DATA_RAW)
        logger.info(f"📦 Dataset descomprimido: {zip_files[0].name}")
else:
    logger.info("📂 Dataset ya existe en cache. Omitiendo descarga.")

In [ ]:
# Celda 2: Carga del Dataset con Tipos Optimizados

# Configuración de muestreo (ajustar según necesidad)
SAMPLE_SIZE = 5000  # 1000 para pruebas, 50000 para producción completa

logger.info(f"🔄 Cargando dataset con muestreo: {SAMPLE_SIZE} filas")

# Encontrar archivo CSV (puede estar en subcarpeta tras descompresión)
csv_files = list(DATA_RAW.rglob("*.csv"))
if not csv_files:
    raise FileNotFoundError(f"❌ No se encontró CSV en {DATA_RAW}")

# Seleccionar el CSV más grande (probablemente el dataset principal)
csv_path = max(csv_files, key=lambda p: p.stat().st_size)
logger.info(f"📄 Archivo detectado: {csv_path.name}")

# Cargar con tipos optimizados para ahorrar memoria
dtypes = {
    'line number': 'int32',
    'review_en': 'string',
    'review_es': 'string',
    'sentiment': 'category',
    'sentimiento': 'category'
}

# Solo usar columnas que existan en el CSV
usecols = [c for c in dtypes.keys() if c in pd.read_csv(csv_path, nrows=0).columns]
logger.info(f"📋 Columnas detectadas: {usecols}")

df = pd.read_csv(csv_path, dtype={k: v for k, v in dtypes.items() if k in usecols}, usecols=usecols)

# Aplicar muestreo estratificado por sentimiento para balance
if SAMPLE_SIZE < len(df):
    if 'sentiment' in df.columns:
        df = df.groupby('sentiment', group_keys=False).apply(
            lambda x: x.sample(int(np.rint(SAMPLE_SIZE * len(x) / len(df))), random_state=42)
        ).sample(frac=1, random_state=42).reset_index(drop=True)
    else:
        df = df.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)
    logger.info(f"✂️ Muestreo aplicado: {len(df)} filas")

# Validar columnas requeridas
assert 'review_es' in df.columns, "❌ Falta columna 'review_es'"

# Eliminar filas con texto vacío
df = df.dropna(subset=['review_es'])
df = df[df['review_es'].str.len() > 10]

# Normalizar ID de reseña: usar 'line number' si existe, sino el índice
if 'line number' not in df.columns:
    df['line number'] = df.index
    logger.info("🆔 Usando índice como review_id (no se encontró 'line number')")

logger.info(f"✅ Dataset cargado: {len(df)} filas | Memoria: {df.memory_usage(deep=True).sum() / 1e6:.2f} MB")
if 'sentiment' in df.columns:
    logger.info(f"📊 Distribución de sentimientos: {dict(df['sentiment'].value_counts())}")

df.head(3)

### Celda 3: Módulo de Extracción Sintáctica (SpaCy)

**Objetivo:** Extraer pares `(Aspecto, Adjetivo)` de cada reseña usando SpaCy con procesamiento por lotes.

**Optimizaciones aplicadas:**
- **`nlp.pipe(batch_size=256)`**: Procesamiento vectorizado, evita bucles `for doc in nlp(textos)`
- **Deshabilitar componentes**: `disable=['ner', 'textcat', 'entity_linker']` para ahorrar CPU
- **Lematización**: Normaliza aspectos (ej: 'actuaciones' → 'actuación') para evitar duplicados
- **Manejo de errores**: Try-except por reseña, loguear sin detener el pipeline

**Lógica de extracción:**
1. Para cada oración, identificar sustantivos (`NOUN`, `PROPN`)
2. Buscar hijos con dependencia `amod` (adjetivo modificador)
3. Extraer lema del sustantivo (aspecto) y forma del adjetivo
4. Guardar fragmento original para contexto del BERT

In [ ]:
# Celda 3: Configuración del Modelo SpaCy

SPACY_MODEL = "es_core_news_md"
BATCH_SIZE_SPACY = 256

logger.info(f"🔧 Cargando modelo SpaCy: {SPACY_MODEL}")

nlp = cargar_modelo_spacy(SPACY_MODEL)

if 'sentencizer' not in nlp.pipe_names:
    nlp.add_pipe('sentencizer')

logger.info(f"✅ Modelo cargado. Componentes activos: {nlp.pipe_names}")

In [ ]:
# Celda 3: Ejecución de la Extracción de Aspectos

gc.collect()

textos = df['review_es'].tolist()
ids = df['line number'].tolist()

logger.info(f"🎯 Iniciando extracción de aspectos para {len(textos)} reseñas...")

aspectos_extraidos = extraer_fragmentos_para_bert(
    textos=textos,
    ids=ids,
    nlp=nlp,
    batch_size=BATCH_SIZE_SPACY,
    incluir_texto_completo=False
)

df_aspectos = pd.DataFrame(aspectos_extraidos)

logger.info(f"📊 Total aspectos extraídos: {len(df_aspectos)}")
logger.info(f"📁 Memoria del DataFrame: {df_aspectos.memory_usage(deep=True).sum() / 1e6:.2f} MB")

print("\n🎭 Top 10 Aspectos más frecuentes:")
print(df_aspectos['aspecto'].value_counts().head(10))

print("\n🌈 Top 10 Adjetivos más frecuentes:")
print(df_aspectos['adjetivo'].value_counts().head(10))

print("\n💡 Ejemplos de extracción:")
df_aspectos[['review_id', 'aspecto', 'adjetivo', 'fragmento']].head(10)

In [ ]:
# Celda 3: Checkpoint Intermedio (Persistencia segura)

checkpoint_spacy = DATA_PROCESSED / "checkpoint_aspectos_spacy.parquet"
df_aspectos.to_parquet(checkpoint_spacy, index=False, compression='snappy')

logger.info(f"💾 Checkpoint guardado: {checkpoint_spacy}")
logger.info(f"📦 Tamaño del archivo: {checkpoint_spacy.stat().st_size / 1e6:.2f} MB")

del textos
del ids
del aspectos_extraidos
gc.collect()

print("✅ Checkpoint guardado. Listo para Celda 4a (Fine-tuning).")

### Celda 4a: Fine-tuning del Modelo BERT

**Objetivo:** Ajustar `pysentimiento/robertuito-sentiment-analysis` al dominio específico de reseñas de IMDb en español.

**¿Por qué fine-tuning?**
- robertuito se entrenó en TASS 2020 (tweets y redes sociales en español)
- IMDb tiene jerga cinematográfica distinta: "actuación", "guion", "banda sonora", "fotografía"
- Fine-tuning adapta el modelo al vocabulario y estilo de las reseñas

**Configuración:**
- **Modelo base:** `pysentimiento/robertuito-sentiment-analysis`
- **Datos:** Reseñas IMDb español con labels `positive`/`negative` (binario)
- **Framework:** `transformers.Trainer` con `accelerate` para GPU
- **Épocas:** 2-3 (~30-60 min en T4)
- **Batch size:** 16 (por VRAM limitada en T4)
- **Salida:** `data/models/robertuito-imdb-finetuned/` en Google Drive

**Output:**
- Pesos del modelo fine-tuneado guardados en Drive
- Métricas de entrenamiento (loss, accuracy, f1)
- El modelo se usará en Celda 4b para inferencia

In [ ]:
# Celda 4a: Preparación del Dataset para Fine-tuning

from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate

# Configuración del fine-tuning
BERT_BASE_MODEL = "pysentimiento/robertuito-sentiment-analysis"
FINETUNED_MODEL_PATH = DATA_MODELS / "robertuito-imdb-finetuned"
FINETUNING_EPOCHS = 3
FINETUNING_BATCH_SIZE = 16
FINETUNING_LEARNING_RATE = 2e-5
FINETUNING_EVAL_RATIO = 0.1  # 10% para validación

logger.info(f"🔧 Preparando fine-tuning de {BERT_BASE_MODEL}")

# Verificar que existan labels en el dataset
if 'sentiment' not in df.columns:
    raise ValueError("❌ El dataset no tiene columna 'sentiment' para fine-tuning")

# Mapear labels a IDs numéricos
label_map = {'positive': 1, 'negative': 0}
label_map_inv = {1: 'positive', 0: 'negative'}

# Preparar dataset para Hugging Face
df_finetune = df[['review_es', 'sentiment']].copy()
df_finetune['labels'] = df_finetune['sentiment'].map(label_map)
df_finetune = df_finetune.dropna(subset=['labels'])
df_finetune['labels'] = df_finetune['labels'].astype(int)
df_finetune = df_finetune.rename(columns={'review_es': 'text'})

# Convertir a Dataset de Hugging Face
dataset = Dataset.from_pandas(df_finetune[['text', 'labels']])

# Split train/eval
dataset = dataset.train_test_split(
    test_size=FINETUNING_EVAL_RATIO,
    seed=42,
    stratify_by_column='labels'
)

logger.info(f"📊 Dataset para fine-tuning: {len(dataset['train'])} train / {len(dataset['test'])} eval")
logger.info(f"🏷️  Distribución train: {dict(pd.Series(dataset['train']['labels']).value_counts())}")

print(f"✅ Dataset preparado: {len(dataset['train'])} ejemplos de entrenamiento")

In [ ]:
# Celda 4a: Tokenización y Carga del Modelo Base

logger.info(f"🔧 Cargando tokenizer y modelo base: {BERT_BASE_MODEL}")

# Cargar tokenizer
tokenizer = AutoTokenizer.from_pretrained(BERT_BASE_MODEL)

# Función de tokenización
def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=128
    )

# Tokenizar dataset
tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Cargar modelo base (2 clases: positive/negative)
model = AutoModelForSequenceClassification.from_pretrained(
    BERT_BASE_MODEL,
    num_labels=2,
    ignore_mismatched_sizes=True  # Ignorar si el modelo base tiene 3 clases
)

logger.info(f"✅ Modelo cargado con {model.num_labels} clases de salida")

# Liberar memoria antes del entrenamiento
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("✅ Modelo y tokenizer listos para entrenamiento")

In [ ]:
# Celda 4a: Configuración del Trainer y Entrenamiento

# Métricas de evaluación
metric_accuracy = evaluate.load("accuracy")
metric_f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = metric_accuracy.compute(predictions=predictions, references=labels)["accuracy"]
    f1 = metric_f1.compute(predictions=predictions, references=labels, average="weighted")["f1"]
    return {"accuracy": accuracy, "f1": f1}

# Argumentos de entrenamiento
training_args = TrainingArguments(
    output_dir=str(DATA_MODELS / "checkpoints"),
    num_train_epochs=FINETUNING_EPOCHS,
    per_device_train_batch_size=FINETUNING_BATCH_SIZE,
    per_device_eval_batch_size=FINETUNING_BATCH_SIZE,
    learning_rate=FINETUNING_LEARNING_RATE,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=50,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),  # Mixed precision si hay GPU
    report_to="none",  # No usar wandb/tensorboard
)

# Inicializar Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    compute_metrics=compute_metrics,
)

logger.info(f"🚀 Iniciando fine-tuning: {FINETUNING_EPOCHS} épocas, batch={FINETUNING_BATCH_SIZE}")

# Entrenar
train_result = trainer.train()

# Guardar métricas
logger.info(f"📊 Métricas finales: {train_result.metrics}")

print("\n" + "="*60)
print("🎉 FINE-TUNING COMPLETADO")
print("="*60)
print(f"📊 Loss final: {train_result.metrics['train_loss']:.4f}")
print(f"📈 Tiempo de entrenamiento: {train_result.metrics['train_runtime']:.1f}s")
print("="*60)

In [ ]:
# Celda 4a: Evaluación y Guardado del Modelo Fine-tuneado

# Evaluar en el set de test
eval_results = trainer.evaluate()

logger.info(f"📊 Resultados de evaluación: {eval_results}")

print("\n📊 MÉTRICAS DE EVALUACIÓN:")
print(f"  Accuracy: {eval_results['eval_accuracy']:.4f}")
print(f"  F1 Score: {eval_results['eval_f1']:.4f}")
print(f"  Loss: {eval_results['eval_loss']:.4f}")

# Guardar modelo fine-tuneado
logger.info(f"💾 Guardando modelo fine-tuneado en: {FINETUNED_MODEL_PATH}")

trainer.save_model(str(FINETUNED_MODEL_PATH))
tokenizer.save_pretrained(str(FINETUNED_MODEL_PATH))

# Guardar configuración de labels
import json
config_path = FINETUNED_MODEL_PATH / "label_config.json"
with open(config_path, 'w') as f:
    json.dump({
        'id2label': label_map_inv,
        'label2id': label_map,
        'base_model': BERT_BASE_MODEL,
        'training_samples': len(dataset['train']),
        'eval_accuracy': eval_results['eval_accuracy'],
        'eval_f1': eval_results['eval_f1']
    }, f, indent=2)

logger.info(f"✅ Modelo guardado: {FINETUNED_MODEL_PATH}")
logger.info(f"📦 Tamaño: {sum(f.stat().st_size for f in FINETUNED_MODEL_PATH.rglob('*') if f.is_file()) / 1e6:.2f} MB")

# Liberar memoria
del trainer
del model
del tokenizer
del tokenized_datasets
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(f"\n✅ Modelo fine-tuneado guardado en: {FINETUNED_MODEL_PATH}")
print("✅ Listo para Celda 4b (Inferencia con modelo fine-tuneado)")

### Celda 4b: Inferencia con Modelo Fine-tuneado

**Objetivo:** Clasificar el sentimiento de cada fragmento extraído usando el modelo fine-tuneado en Celda 4a.

**Optimizaciones aplicadas:**
- **`pipeline(device=0)`**: Fuerza uso de GPU (CUDA)
- **`batch_size=64` o `128`**: Procesamiento por lotes para saturar la GPU
- **`truncation=True, max_length=128`**: Evita exceder límite de tokens del modelo
- **`torch.cuda.empty_cache()`**: Libera VRAM entre lotes grandes para evitar OOM
- **Manejo de errores**: Si un batch falla, se salta y continúa (no se pierden los anteriores)

**Mapeo de labels:**
- `LABEL_1` → positivo
- `LABEL_0` → negativo

**Nota:** Si el modelo fine-tuneado no existe en Drive, se usará el modelo base como fallback.

In [ ]:
# Celda 4b: Configuración del Pipeline BERT con Modelo Fine-tuneado

from transformers import pipeline

# Configuración del modelo BERT para sentimiento
FINETUNED_MODEL_PATH = DATA_MODELS / "robertuito-imdb-finetuned"
BATCH_SIZE_BERT = 64  # Ajustar según VRAM disponible (64 para T4, 128 para A100)
MAX_LENGTH = 128

# Determinar qué modelo usar
if FINETUNED_MODEL_PATH.exists() and (FINETUNED_MODEL_PATH / "config.json").exists():
    BERT_MODEL = str(FINETUNED_MODEL_PATH)
    logger.info(f"✅ Usando modelo fine-tuneado: {BERT_MODEL}")
    
    # Cargar configuración de labels
    import json
    with open(FINETUNED_MODEL_PATH / "label_config.json", 'r') as f:
        label_config = json.load(f)
    sentiment_map = {f"LABEL_{k}": v for k, v in label_config['id2label'].items()}
    print(f"📊 Métricas del modelo fine-tuneado: Accuracy={label_config['eval_accuracy']:.4f}, F1={label_config['eval_f1']:.4f}")
else:
    BERT_MODEL = "pysentimiento/robertuito-sentiment-analysis"
    logger.warning(f"⚠️ Modelo fine-tuneado no encontrado. Usando modelo base: {BERT_MODEL}")
    sentiment_map = {
        "POS": "positive",
        "NEG": "negative",
        "NEU": "neutral"
    }

# Verificar GPU antes de cargar modelo
if torch.cuda.is_available():
    device = 0
    print(f"🚀 Usando GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 VRAM disponible: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    device = -1
    print("⚠️ GPU no disponible. Usando CPU (será más lento).")

logger.info(f"🔧 Cargando modelo BERT: {BERT_MODEL}")

# Configurar pipeline de inferencia
classifier = configurar_pipeline_sentimiento(
    modelo=BERT_MODEL,
    device=device,
    batch_size=BATCH_SIZE_BERT,
    max_length=MAX_LENGTH
)

print("✅ Pipeline BERT configurado correctamente.")

In [ ]:
# Celda 4b: Ejecución de Inferencia BERT

# Cargar checkpoint de SpaCy si no está en memoria
if 'df_aspectos' not in globals():
    checkpoint_spacy = DATA_PROCESSED / "checkpoint_aspectos_spacy.parquet"
    if checkpoint_spacy.exists():
        df_aspectos = pd.read_parquet(checkpoint_spacy)
        logger.info(f"📂 Checkpoint SpaCy cargado: {len(df_aspectos)} filas")
    else:
        raise FileNotFoundError("❌ No se encontró checkpoint de SpaCy. Ejecutar Celda 3 primero.")

# Preparar fragmentos para BERT
fragmentos = df_aspectos['fragmento'].tolist()

logger.info(f"🎯 Enviando {len(fragmentos)} fragmentos al modelo BERT...")

# Ejecutar inferencia usando función del módulo src/
resultados_sentimiento = analizar_sentimiento_lote(
    fragmentos=fragmentos,
    classifier=classifier,
    batch_size=BATCH_SIZE_BERT
)

# Crear DataFrame de resultados BERT
df_sentimientos = pd.DataFrame(resultados_sentimiento)
df_sentimientos['sentimiento'] = df_sentimientos['label_original'].map(sentiment_map).fillna('unknown')

# Unir con aspectos
df_aspectos['sentimiento_bert'] = df_sentimientos['sentimiento']
df_aspectos['confianza'] = df_sentimientos['confianza']
df_aspectos['label_original'] = df_sentimientos['label_original']

# Mostrar distribución de sentimientos
print("\n📊 Distribución de Sentimientos por Aspecto:")
print(df_aspectos['sentimiento_bert'].value_counts())

print("\n💡 Ejemplos con sentimiento:")
print(df_aspectos[['aspecto', 'adjetivo', 'fragmento', 'sentimiento_bert', 'confianza']].head(10))

# Liberar memoria
del fragmentos
del resultados_sentimiento
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

logger.info("✅ Celda 4b completada. Memoria liberada.")

In [ ]:
# Celda 4b: Checkpoint BERT (Persistencia)

checkpoint_bert = DATA_PROCESSED / "checkpoint_aspectos_sentimientos.parquet"
df_aspectos.to_parquet(checkpoint_bert, index=False, compression='snappy')

logger.info(f"💾 Checkpoint BERT guardado: {checkpoint_bert}")
logger.info(f"📦 Tamaño del archivo: {checkpoint_bert.stat().st_size / 1e6:.2f} MB")

print("✅ Checkpoint BERT guardado. Listo para Celda 5 (Consolidación Final).")

### Celda 5: Orquestación Final y Guardado

**Objetivo:** Consolidar todos los resultados, calcular métricas agregadas por aspecto y exportar el archivo final optimizado para Streamlit.

**Procesos realizados:**
1. **Merge con datos originales**: Unir `sentiment` original del dataset con predicciones BERT
2. **Agregación por aspecto**: Calcular métricas desglosadas (conteo, confianza promedio, sentimiento dominante)
3. **Normalización**: Eliminar aspectos poco frecuentes (ruido) y estandarizar formato
4. **Exportación final**: Parquet comprimido con todas las columnas necesarias para el dashboard

**Estructura del output final (`aspectos_sentimientos_final.parquet`):**
- `review_id`: ID original de la reseña
- `aspecto`: Sustantivo lematizado (aspecto extraído)
- `adjetivo`: Adjetivo tal cual aparece en el texto
- `fragmento`: Contexto completo del par aspecto-adjetivo
- `sentimiento_bert`: Etiqueta predicha (positive/negative)
- `confianza`: Score de confianza del modelo BERT
- `sentiment_original`: Polaridad original del dataset (pos/neg)
- `metricas_aspecto`: Conteo, confianza promedio y sentimiento dominante por aspecto

In [ ]:
# Celda 5: Consolidación Final y Exportación

# Cargar checkpoint si no está en memoria
if 'df_aspectos' not in globals():
    checkpoint_bert = DATA_PROCESSED / "checkpoint_aspectos_sentimientos.parquet"
    if checkpoint_bert.exists():
        df_aspectos = pd.read_parquet(checkpoint_bert)
        logger.info(f"📂 Checkpoint BERT cargado: {len(df_aspectos)} filas")
    else:
        raise FileNotFoundError("❌ No se encontró checkpoint BERT. Ejecutar Celda 4b primero.")

# ───────────────────────────────────────────────
# PASO 1: Merge con datos originales del dataset
# ───────────────────────────────────────────────

logger.info("🔄 Iniciando consolidación final...")

if 'df' in globals():
    cols_origen = ['line number']
    if 'sentiment' in df.columns:
        cols_origen.append('sentiment')
    if 'sentimiento' in df.columns:
        cols_origen.append('sentimiento')
    
    df_original = df[cols_origen].copy()
    df_original.columns = ['review_id'] + [f'sentiment_{c}' for c in cols_origen[1:]]
    
    df_final = df_aspectos.merge(
        df_original,
        on='review_id',
        how='left'
    )
else:
    logger.warning("⚠️ DataFrame original no disponible. Continuando sin merge.")
    df_final = df_aspectos.copy()
    df_final['sentiment_original'] = 'unknown'

# ───────────────────────────────────────────────
# PASO 2: Agregación de métricas por aspecto
# ───────────────────────────────────────────────

logger.info("📊 Calculando métricas agregadas por aspecto...")

metricas_aspecto = df_final.groupby('aspecto').agg(
    total_apariciones=('aspecto', 'size'),
    confianza_promedio=('confianza', 'mean'),
    sentimiento_dominante=('sentimiento_bert', lambda x: x.mode().iloc[0] if not x.mode().empty else 'neutral'),
    pct_positivo=('sentimiento_bert', lambda x: (x == 'positive').mean() * 100),
    pct_negativo=('sentimiento_bert', lambda x: (x == 'negative').mean() * 100)
).reset_index()

# Filtrar aspectos con poca frecuencia (ruido): mínimo 5 apariciones
MIN_FRECUENCIA = 5
aspectos_validos = metricas_aspecto[metricas_aspecto['total_apariciones'] >= MIN_FRECUENCIA]['aspecto'].tolist()

df_final = df_final[df_final['aspecto'].isin(aspectos_validos)].copy()
metricas_aspecto = metricas_aspecto[metricas_aspecto['total_apariciones'] >= MIN_FRECUENCIA].copy()

logger.info(f"🎯 Aspectos filtrados (>= {MIN_FRECUENCIA} apariciones): {len(aspectos_validos)}")

# ───────────────────────────────────────────────
# PASO 3: Optimizar tipos de datos para exportación
# ───────────────────────────────────────────────

df_final['sentimiento_bert'] = df_final['sentimiento_bert'].astype('category')
df_final['aspecto'] = df_final['aspecto'].astype('category')
df_final['confianza'] = df_final['confianza'].astype('float32')

# ───────────────────────────────────────────────
# PASO 4: Exportar resultado final a Parquet
# ───────────────────────────────────────────────

OUTPUT_FILE = DATA_PROCESSED / "aspectos_sentimientos_final.parquet"
METRICAS_FILE = DATA_PROCESSED / "metricas_por_aspecto.parquet"

df_final.to_parquet(OUTPUT_FILE, index=False, compression='snappy')
logger.info(f"💾 Archivo final guardado: {OUTPUT_FILE}")
logger.info(f"📦 Tamaño: {OUTPUT_FILE.stat().st_size / 1e6:.2f} MB")

metricas_aspecto.to_parquet(METRICAS_FILE, index=False, compression='snappy')
logger.info(f"💾 Métricas por aspecto guardadas: {METRICAS_FILE}")

# ───────────────────────────────────────────────
# RESUMEN FINAL
# ───────────────────────────────────────────────

print("\n" + "="*60)
print("🎉 PIPELINE ABSA COMPLETADO CON ÉXITO")
print("="*60)
print(f"📊 Total de reseñas procesadas: {df_final['review_id'].nunique()}")
print(f"🎭 Total de pares aspecto-adjetivo: {len(df_final)}")
print(f"🏷️  Aspectos únicos encontrados: {df_final['aspecto'].nunique()}")
print(f"📁 Archivo final: {OUTPUT_FILE.name}")
print(f"📈 Métricas por aspecto: {METRICAS_FILE.name}")
print(f"🧠 Modelo fine-tuneado: {FINETUNED_MODEL_PATH}")
print("="*60)

print("\n📊 Distribución de Sentimientos (BERT):")
print(df_final['sentimiento_bert'].value_counts())

print("\n🎭 Top 10 Aspectos más mencionados:")
print(df_final['aspecto'].value_counts().head(10))

print("\n🌈 Top 10 Adjetivos más frecuentes:")
print(df_final['adjetivo'].value_counts().head(10))

print("\n💡 Vista previa del output final:")
print(df_final[['review_id', 'aspecto', 'adjetivo', 'sentimiento_bert', 'confianza']].head())

# ───────────────────────────────────────────────
# LIMPIEZA FINAL DE MEMORIA
# ───────────────────────────────────────────────

logger.info("🧹 Limpiando memoria...")
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

logger.info("✅ Pipeline finalizado. Todo listo para Streamlit! 🚀")

### 🎉 Pipeline Completado

**Archivos generados:**

**`data/processed/`:**
1. `aspectos_sentimientos_final.parquet` — Dataset completo con aspectos y sentimientos (listo para Streamlit)
2. `metricas_por_aspecto.parquet` — Métricas agregadas por aspecto
3. `checkpoint_aspectos_spacy.parquet` — Checkpoint intermedio de SpaCy
4. `checkpoint_aspectos_sentimientos.parquet` — Checkpoint intermedio de BERT

**`data/models/`:**
5. `robertuito-imdb-finetuned/` — Pesos del modelo fine-tuneado (~500MB)
   - `config.json` — Configuración del modelo
   - `model.safetensors` — Pesos del modelo
   - `tokenizer.json` — Tokenizer
   - `label_config.json` — Mapeo de labels y métricas de evaluación

**Próximos pasos:**
- Descargar los archivos Parquet y el modelo desde Google Drive
- Construir dashboard de Streamlit consumiendo `aspectos_sentimientos_final.parquet`
- Usar el modelo fine-tuneado para inferencia en producción
- Visualizar: distribución de aspectos, nubes de adjetivos, evolución de sentimiento por review
